<a href="https://colab.research.google.com/github/JC9427/JC-programming/blob/main/HW1_Part2_(%E6%AD%B7%E5%8F%B2%E8%B3%87%E6%96%99)%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **日常支出速算與分攤（作業一）**
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分
頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer
- 歷史資料、設定按鈕



GoogleSheet: https://docs.google.com/spreadsheets/d/1jVPEMqCki2cnHLe3H6uumymnpc9G3JdVRLflp7mmny8/edit?usp=sharing

In [ ]:
import gradio as gr
import pandas as pd
import datetime
import gspread
from google.colab import auth
from google.auth import default

# --- 1. 設定與語言字典 ---
SHEET_URL = "https://docs.google.com/spreadsheets/d/1jVPEMqCki2cnHLe3H6uumymnpc9G3JdVRLflp7mmny8/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"
REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人", "地點", "支付方式", "備註"]

LANG_DICT = {
    "繁體中文": {
        "settings_btn": "⚙️ 設定",
        "settings_title": "### ⚙️ 應用程式設定",
        "lbl_lang_select": "語言切換",
        "lbl_theme_select": "介面主題",
        "btn_close": "關閉",
        "title": "## 🧾 日常支出速算與分攤",
        "tab_add": "➕ 新增支出", "tab_sum": "📊 彙總 / AA 分攤", "tab_raw": "📒 檢視原始資料",
        "lbl_date": "日期 YYYY-MM-DD", "lbl_time": "時間 HH:MM",
        "lbl_cat": "分類", "lbl_item": "品項", "lbl_amt": "金額",
        "lbl_payer": "付款人", "lbl_loc": "地點", "lbl_method": "支付方式", "lbl_notes": "備註",
        "btn_add": "新增到工作表", "lbl_total": "目前總額", "lbl_cat_df": "分類小計",
        "lbl_settle_df": "AA 分攤結算", "btn_refresh": "讀取最新彙總", "btn_view": "讀取資料",
        "theme_opts": ["系統自訂", "白天", "黑夜"]
    },
    "English": {
        "settings_btn": "⚙️ Settings",
        "settings_title": "### ⚙️ App Settings",
        "lbl_lang_select": "Language",
        "lbl_theme_select": "Theme",
        "btn_close": "Close",
        "title": "## 🧾 Expense Tracker",
        "tab_add": "➕ Add Expense", "tab_sum": "📊 Summary / AA", "tab_raw": "📒 Raw Data",
        "lbl_date": "Date YYYY-MM-DD", "lbl_time": "Time HH:MM",
        "lbl_cat": "Category", "lbl_item": "Item", "lbl_amt": "Amount",
        "lbl_payer": "Payer", "lbl_loc": "Location", "lbl_method": "Method", "lbl_notes": "Notes",
        "btn_add": "Add to Sheet", "lbl_total": "Total Amount", "lbl_cat_df": "Category Summary",
        "lbl_settle_df": "Settlement", "btn_refresh": "Refresh Summary", "btn_view": "View Data",
        "theme_opts": ["System", "Light", "Dark"]
    }
}

# --- 2. JavaScript & CSS (控制主題與右上角按鈕) ---
theme_js = """
(v) => {
    const el = document.querySelector('body');
    if (v === '黑夜' || v === 'Dark') { el.classList.add('dark'); }
    else if (v === '白天' || v === 'Light') { el.classList.remove('dark'); }
    else {
        if (window.matchMedia('(prefers-color-scheme: dark)').matches) { el.classList.add('dark'); }
        else { el.classList.remove('dark'); }
    }
}
"""

css = """
#settings_btn {
    position: fixed; top: 15px; right: 15px; z-index: 1001;
    width: auto !important; padding: 5px 15px;
    background-color: #f0f0f0; border: 1px solid #ddd;
    border-radius: 20px; cursor: pointer; transition: all 0.3s ease;
}
.dark #settings_btn { background-color: #333; border-color: #555; color: white; }
#settings_btn:hover {
    background-color: #ff8c00 !important;
    color: white !important;
    border-color: #ff8c00;
}
#settings_panel {
    position: fixed; top: 60px; right: 15px; z-index: 1000;
    background: white; padding: 20px; border: 1px solid #ddd;
    border-radius: 10px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); width: 280px;
}
.dark #settings_panel { background: #1f1f1f; border-color: #444; color: white; }
footer {display: none !important;}
"""

# --- 3. 核心後端邏輯 ---
_auth_done, _gc, _ws = False, None, None

def _ensure_auth_and_ws():
    global _auth_done, _gc, _ws
    if not _auth_done:
        auth.authenticate_user(); creds, _ = default()
        _gc = gspread.authorize(creds); _auth_done = True
    if _ws is None:
        gs = _gc.open_by_url(SHEET_URL); _ws = gs.worksheet(WORKSHEET_NAME)
        _ensure_headers()
    return _ws

def _ensure_headers():
    rows = _ws.get_all_values()
    if not rows:
        _ws.append_row(REQUIRED_COLUMNS, value_input_option="USER_ENTERED")
        return
    header = rows[0]
    if header != REQUIRED_COLUMNS:
        existing = {h: idx for idx, h in enumerate(header)}
        if len(rows) > 1:
            new_rows = []
            for r in rows[1:]:
                mapping = {col: (r[existing[col]] if col in existing and existing[col] < len(r) else "")
                           for col in REQUIRED_COLUMNS}
                new_rows.append([mapping[c] for c in REQUIRED_COLUMNS])
            _ws.resize(rows=1)
            _ws.update('1:1', [REQUIRED_COLUMNS])
            _ws.append_rows(new_rows, value_input_option="USER_ENTERED")
        else:
            _ws.update('1:1', [REQUIRED_COLUMNS])

def _read_df():
    ws = _ensure_auth_and_ws(); values = ws.get_all_values()
    if not values or len(values) < 2: return pd.DataFrame(columns=REQUIRED_COLUMNS)
    df = pd.DataFrame(values[1:], columns=values[0])
    if "金額" in df.columns: df["金額"] = pd.to_numeric(df["金額"], errors="coerce").fillna(0.0)
    return df

def get_history_choices():
    df = _read_df()
    if df.empty: return [], [], [], [], []
    def _u(c): return sorted([str(x).strip() for x in df[c].unique() if x and str(x).strip()]) if c in df.columns else []
    return _u("分類"), _u("付款人"), _u("地點"), _u("支付方式"), _u("品項")

def _make_summary_tables(df):
    by_cat = df.groupby("分類", as_index=False)["金額"].sum().sort_values("金額", ascending=False) if "分類" in df.columns else pd.DataFrame(columns=["分類", "金額"])
    if "付款人" in df.columns and not df.empty:
        df["付款人"] = df["付款人"].replace("", "匿名")
        total = df["金額"].sum()
        payers = sorted(df["付款人"].unique())
        aa_share = total / max(len(payers), 1)
        paid_by = df.groupby("付款人", as_index=False)["金額"].sum().rename(columns={"金額": "實付"})
        settle = pd.DataFrame({"付款人": payers}).merge(paid_by, on="付款人", how="left").fillna(0)
        settle["應付(AA)"] = aa_share
        settle["差額(實付-應付)"] = settle["實付"] - settle["應付(AA)"]
        settle = settle.sort_values("差額(實付-應付)", ascending=False)
    else:
        settle = pd.DataFrame(columns=["付款人", "實付", "應付(AA)", "差額(實付-應付)"])
    return by_cat, settle

# --- 4. 介面互動與切換函數 ---

def toggle_settings(visible): return gr.update(visible=not visible), not visible

def change_language(lang):
    d = LANG_DICT[lang]
    return [
        gr.update(value=d["settings_btn"]), gr.update(value=d["settings_title"]),
        gr.update(label=d["lbl_lang_select"]), gr.update(label=d["lbl_theme_select"], choices=d["theme_opts"]),
        gr.update(value=d["btn_close"]), gr.update(value=d["title"]),
        gr.update(label=d["tab_add"]), gr.update(label=d["tab_sum"]), gr.update(label=d["tab_raw"]),
        gr.update(label=d["lbl_date"]), gr.update(label=d["lbl_time"]),
        gr.update(label=d["lbl_cat"]), gr.update(label=d["lbl_item"]),
        gr.update(label=d["lbl_amt"]), gr.update(label=d["lbl_payer"]),
        gr.update(label=d["lbl_loc"]), gr.update(label=d["lbl_method"]),
        gr.update(label=d["lbl_notes"]), gr.update(value=d["btn_add"]),
        gr.update(label=d["lbl_total"]), gr.update(label=d["lbl_cat_df"]),
        gr.update(label=d["lbl_settle_df"]), gr.update(value=d["btn_refresh"]),
        gr.update(value=d["btn_view"])
    ]

def add_expense(date_str, time_str, category, item, amount, payer, location, method, notes):
    try:
        if not date_str: date_str = datetime.date.today().strftime("%Y-%m-%d")
        amount_val = float(amount)
        ws = _ensure_auth_and_ws()
        ws.append_row([date_str, time_str or "", category, item, amount_val, payer, location, method, notes], value_input_option="USER_ENTERED")
        df = _read_df()
        cat_sum, settle = _make_summary_tables(df)
        total = float(df["金額"].sum())
        c_opts, p_opts, l_opts, m_opts, i_opts = get_history_choices()
        return (f"✅ Added: {item}", total, cat_sum, settle,
                gr.update(choices=c_opts, value=None), gr.update(choices=p_opts, value=None),
                gr.update(choices=l_opts, value=None), gr.update(choices=m_opts, value=None),
                gr.update(choices=i_opts, value=None), "", "", "")
    except Exception as e: return f"❌ Error: {e}", 0, None, None, gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), "", "", ""

def refresh_summary():
    df = _read_df()
    total = float(df["金額"].sum()) if not df.empty else 0.0
    by_cat, settle = _make_summary_tables(df)
    return "✅ Updated", total, by_cat, settle, df

# --- 5. Gradio 介面配置 ---

try:
    init_c, init_p, init_l, init_m, init_i = get_history_choices()
except:
    init_c, init_p, init_l, init_m, init_i = [], [], [], [], []

with gr.Blocks(title="日常支出速算與分攤", css=css) as demo:
    settings_visible = gr.State(False)

    # 右上角按鈕
    settings_btn = gr.Button(LANG_DICT["繁體中文"]["settings_btn"], elem_id="settings_btn")

    # 設定面板
    with gr.Column(elem_id="settings_panel", visible=False) as settings_panel:
        settings_md = gr.Markdown(LANG_DICT["繁體中文"]["settings_title"])
        lang_drop = gr.Dropdown(label=LANG_DICT["繁體中文"]["lbl_lang_select"], choices=["繁體中文", "English"], value="繁體中文")
        theme_drop = gr.Dropdown(label=LANG_DICT["繁體中文"]["lbl_theme_select"], choices=LANG_DICT["繁體中文"]["theme_opts"], value="系統自訂")
        close_btn = gr.Button(LANG_DICT["繁體中文"]["btn_close"], size="sm")

    main_title = gr.Markdown(LANG_DICT["繁體中文"]["title"])

    with gr.Tabs() as tabs:
        with gr.Tab(LANG_DICT["繁體中文"]["tab_add"]) as tab1:
            with gr.Row():
                date_in = gr.Textbox(label=LANG_DICT["繁體中文"]["lbl_date"], value=datetime.date.today().strftime("%Y-%m-%d"))
                time_in = gr.Textbox(label=LANG_DICT["繁體中文"]["lbl_time"])
            with gr.Row():
                cat_in = gr.Dropdown(label=LANG_DICT["繁體中文"]["lbl_cat"], choices=init_c, allow_custom_value=True)
                item_in = gr.Dropdown(label=LANG_DICT["繁體中文"]["lbl_item"], choices=init_i, allow_custom_value=True)
            with gr.Row():
                amt_in = gr.Textbox(label=LANG_DICT["繁體中文"]["lbl_amt"])
                payer_in = gr.Dropdown(label=LANG_DICT["繁體中文"]["lbl_payer"], choices=init_p, allow_custom_value=True)
            with gr.Row():
                loc_in = gr.Dropdown(label=LANG_DICT["繁體中文"]["lbl_loc"], choices=init_l, allow_custom_value=True)
                meth_in = gr.Dropdown(label=LANG_DICT["繁體中文"]["lbl_method"], choices=init_m, allow_custom_value=True)
                note_in = gr.Textbox(label=LANG_DICT["繁體中文"]["lbl_notes"])
            add_btn = gr.Button(LANG_DICT["繁體中文"]["btn_add"], variant="primary")
            add_msg = gr.Markdown()
            total_out = gr.Number(label=LANG_DICT["繁體中文"]["lbl_total"], interactive=False)
            cat_df = gr.Dataframe(label=LANG_DICT["繁體中文"]["lbl_cat_df"])
            settle_df = gr.Dataframe(label=LANG_DICT["繁體中文"]["lbl_settle_df"])

        with gr.Tab(LANG_DICT["繁體中文"]["tab_sum"]) as tab2:
            refresh_btn = gr.Button(LANG_DICT["繁體中文"]["btn_refresh"])
            total2 = gr.Number(label=LANG_DICT["繁體中文"]["lbl_total"])
            cat_df2 = gr.Dataframe(label=LANG_DICT["繁體中文"]["lbl_cat_df"])
            settle_df2 = gr.Dataframe(label=LANG_DICT["繁體中文"]["lbl_settle_df"])
            raw_preview = gr.Dataframe()

        with gr.Tab(LANG_DICT["繁體中文"]["tab_raw"]) as tab3:
            view_btn = gr.Button(LANG_DICT["繁體中文"]["btn_view"])
            view_df = gr.Dataframe()

    # --- 互動事件 ---
    settings_btn.click(toggle_settings, inputs=[settings_visible], outputs=[settings_panel, settings_visible])
    close_btn.click(toggle_settings, inputs=[settings_visible], outputs=[settings_panel, settings_visible])

    lang_drop.change(change_language, inputs=[lang_drop], outputs=[
        settings_btn, settings_md, lang_drop, theme_drop, close_btn,
        main_title, tab1, tab2, tab3, date_in, time_in, cat_in, item_in,
        amt_in, payer_in, loc_in, meth_in, note_in, add_btn,
        total_out, cat_df, settle_df, refresh_btn, view_btn
    ])

    theme_drop.change(fn=None, inputs=[theme_drop], js=theme_js)

    add_btn.click(
        add_expense,
        inputs=[date_in, time_in, cat_in, item_in, amt_in, payer_in, loc_in, meth_in, note_in],
        outputs=[add_msg, total_out, cat_df, settle_df, cat_in, payer_in, loc_in, meth_in, item_in, time_in, amt_in, note_in]
    )

    refresh_btn.click(fn=refresh_summary, inputs=[], outputs=[gr.Markdown(), total2, cat_df2, settle_df2, raw_preview])
    view_btn.click(fn=lambda: _read_df(), outputs=[view_df])

demo.launch(share=True)

/tmp/ipykernel_652/3061689825.py:193: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="日常支出速算與分攤", css=css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cdef5537d6e3e8fd42.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
